# ISMIP6 Coverage Summary

Summarizes the Globus inventory, processing status, variable availability, failures, and basic output-file sanity checks for the ISMIP6 point-subset workflow.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json

import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd


RHO_ICE_KG_M3 = 917.0
SECONDS_PER_YEAR = 365.25 * 24 * 60 * 60

ROOT = Path.cwd()
if not (ROOT / "output" / "ismip6").exists():
    ROOT = ROOT.parent

OUTPUT_ROOT = ROOT / "output" / "ismip6"
SUMMARY_PATH = OUTPUT_ROOT / "ismip6_processing_summary.json"
INVENTORY_PATH = OUTPUT_ROOT / "ismip6_inventory.json"

assert SUMMARY_PATH.exists(), f"Missing summary: {SUMMARY_PATH}"
assert INVENTORY_PATH.exists(), f"Missing inventory: {INVENTORY_PATH}"

with SUMMARY_PATH.open() as f:
    summary = json.load(f)
with INVENTORY_PATH.open() as f:
    inventory = json.load(f)

records = pd.DataFrame(summary["records"])
runs = pd.DataFrame(inventory["runs"])

usable_statuses = {"processed", "skipped_existing"}
records["output_exists"] = records["output_netcdf"].apply(lambda p: Path(p).exists() if isinstance(p, str) else False)
records["usable_output"] = records["status"].isin(usable_statuses) & records["output_exists"]

print("Inventory runs:", inventory["run_count"])
print("Processing records:", summary["record_count"])
print("Created at:", summary["created_at"])
print("Status counts:")
print(records["status"].value_counts().to_string())


In [ ]:
availability = []
for run in inventory["runs"]:
    for variable in inventory["variables"]:
        file_info = run["files"].get(variable)
        availability.append(
            {
                "group": run["group"],
                "model": run["model"],
                "experiment": run["experiment"],
                "standard_variable": variable,
                "available_in_inventory": file_info is not None,
                "remote_variable_hint": None if file_info is None else file_info.get("remote_variable_hint"),
                "filename": None if file_info is None else file_info.get("filename"),
            }
        )

availability = pd.DataFrame(availability)
availability_by_variable = (
    availability.groupby("standard_variable")["available_in_inventory"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "available_files", "count": "runs"})
)
availability_by_variable["missing_files"] = availability_by_variable["runs"] - availability_by_variable["available_files"]
availability_by_variable


In [ ]:
status_by_variable = (
    records.pivot_table(
        index="standard_variable",
        columns="status",
        values="group",
        aggfunc="count",
        fill_value=0,
    )
    .sort_index()
)
status_by_variable


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

availability_by_variable[["available_files", "missing_files"]].plot.bar(
    stacked=True,
    ax=axes[0],
    color=["#2f7f5f", "#c86b4a"],
)
axes[0].set_title("Inventory availability by variable")
axes[0].set_xlabel("")
axes[0].set_ylabel("run-variable files")
axes[0].legend(frameon=False)

status_by_variable.plot.bar(stacked=True, ax=axes[1])
axes[1].set_title("Processing status by variable")
axes[1].set_xlabel("")
axes[1].set_ylabel("records")
axes[1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

for ax in axes:
    ax.grid(True, axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


In [ ]:
run_keys = ["group", "model", "experiment"]
complete = (
    records[records["usable_output"]]
    .groupby(run_keys)["standard_variable"]
    .nunique()
    .reset_index(name="usable_variable_count")
)
complete["complete_four_variable_run"] = complete["usable_variable_count"] == 4

coverage_by_group_model = (
    records.assign(usable=records["usable_output"].astype(int))
    .pivot_table(
        index=["group", "model"],
        columns="standard_variable",
        values="usable",
        aggfunc="sum",
        fill_value=0,
    )
)
coverage_by_group_model["complete_runs"] = complete[complete["complete_four_variable_run"]].groupby(["group", "model"]).size()
coverage_by_group_model["complete_runs"] = coverage_by_group_model["complete_runs"].fillna(0).astype(int)
coverage_by_group_model.sort_index()


In [ ]:
failed = records[records["status"].eq("failed")].copy()
if failed.empty:
    print("No failed records.")
else:
    display(
        failed[
            [
                "group",
                "model",
                "experiment",
                "standard_variable",
                "remote_variable",
                "remote_path",
                "message",
            ]
        ].sort_values(["group", "model", "experiment", "standard_variable"])
    )


In [ ]:
missing = records[records["status"].eq("skipped_missing")].copy()
print("Skipped missing records:", len(missing))
if not missing.empty:
    display(
        missing.pivot_table(
            index=["group", "model"],
            columns="standard_variable",
            values="experiment",
            aggfunc="count",
            fill_value=0,
        )
    )


In [ ]:
bad_outputs = records[records["status"].isin(usable_statuses) & ~records["output_exists"]]
assert bad_outputs.empty, "Some processed/skipped-existing records point to missing output files"

sample_path = Path(records.loc[records["usable_output"], "output_netcdf"].iloc[0])
with netCDF4.Dataset(sample_path) as ds:
    print("Sample file:", sample_path)
    print("dimensions:", {name: len(dim) for name, dim in ds.dimensions.items()})
    print("variables:", list(ds.variables))
    print("point names:", [str(v) for v in ds.variables["point_name"][:]])
    print("time units:", getattr(ds.variables["time"], "units", None))
    for name in ds.variables:
        if name not in {"time", "point", "point_name", "requested_latitude", "requested_longitude", "requested_x", "requested_y", "latitude", "longitude", "x", "y"}:
            print("data variable:", name, ds.variables[name].shape, getattr(ds.variables[name], "units", None))
            break
